## Imports

In [1]:
from pynq.lib.video import *
import numpy as np
import threading
import time
import cv2
import socket
import os

from pynq.overlays.base import BaseOverlay
from datetime import datetime
from pynq.lib import RGBLED
base = BaseOverlay("base.bit")
btns = base.btns_gpio
rgb_led = RGBLED(4)

## Functions and set up

In [2]:
def clear_gpios():
    rgb_led.off()

In [3]:
def write_rgb_led(r, g, b):
    """
    rgb values for the on board RGB LED
    """
    
    color = (r << 0) | (g << 1) | (b << 2)

    rgb_led.write(color)

In [4]:
# set led functions, incorporate this to sample the state every 0.1s
def set_led_blue():
    write_rgb_led(1,0,0)

def set_led_red():
    write_rgb_led(0,0,1)

def set_led_green():
    write_rgb_led(0,1,0)

In [5]:
def get_region(frame, x1, y1, x2, y2):
    return {
        "box": (x1, y1, x2, y2),
        "region": frame[y1:y2, x1:x2]
    }

In [6]:
# get color of skin, hair, eyes, etc
def get_color(region):
    pixels = region.reshape((-1, 3))
    
    if len(pixels) == 0:
        return (0, 0, 0)

    avg_color = np.mean(pixels, axis=0)
    return tuple(avg_color.astype(int))

In [16]:
def color_correct(frame, face_box):
    """
    color correction is applied only to the face region for more consistent skin/hair/eye color
    """
    x, y, w, h = face_box
    edge = 100
    x = int(x) - edge
    y = int(y) - edge
    w = int(w) + 2*edge
    h = int(h) + 2*edge
    roi = frame[y:y+h, x:x+w].copy()

    # LAB normalization
    lab = cv2.cvtColor(roi, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l_norm = cv2.normalize(l, None, 0, 255, cv2.NORM_MINMAX)
    lab_corrected = cv2.merge((l_norm, a, b))
    roi_corrected = cv2.cvtColor(lab_corrected, cv2.COLOR_LAB2BGR)

    # glare handling
    glare_thresh = 220
    hsv = cv2.cvtColor(roi_corrected, cv2.COLOR_BGR2HSV)
    h_ch, s_ch, v_ch = cv2.split(hsv)

    glare_mask = v_ch > glare_thresh

    v_blur = cv2.medianBlur(v_ch, 5)
    v_ch[glare_mask] = v_blur[glare_mask]

    hsv_corrected = cv2.merge((h_ch, s_ch, v_ch))
    roi_final = cv2.cvtColor(hsv_corrected, cv2.COLOR_HSV2BGR)

    # put face region back into original image
    frame[y:y+h, x:x+w] = roi_final

    return frame

In [8]:
def extract_regions(frame, x, y, w, h):
    # eye detection model lol
    eye_cascade = cv2.CascadeClassifier("/home/xilinx/jupyter_notebooks/base/video/data/haarcascade_eye.xml")

    left_cheek = get_region(
        frame,
        x + int(0.20*w),
        y + int(0.55*h),
        x + int(0.30*w),
        y + int(0.80*h)
    )

    right_cheek = get_region(
        frame,
        x + int(0.70*w),
        y + int(0.55*h),
        x + int(0.80*w),
        y + int(0.80*h)
    )

    hair = get_region(
        frame,
        x + int(0.30*w),
        max(0, y - int(0.20*h)),
        x + int(0.70*w),
        y - int(0.03*h)
    )

    face_roi = frame[y:y+h, x:x+w]
    gray_roi = cv2.cvtColor(face_roi, cv2.COLOR_BGR2GRAY)

    eyes = eye_cascade.detectMultiScale(
        gray_roi,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(20,20)
    )

    best_eye = None
    best_score = 0
    
    for (ex, ey, ew, eh) in eyes:
        # remove small boxes (could be a nose)
        if ey > h * 0.5:
            continue

        score = ew * eh
        if score > best_score:
            best_score = score
            best_eye = (ex, ey, ew, eh)


    eye_region = None

    if best_eye is not None:
        ex, ey, ew, eh = best_eye

        eye = get_region(
            frame,
            x + ex,
            y + ey,
            x + ex + ew,
            y + ey + eh
        )

    eye_region = eye

    return left_cheek, right_cheek, hair, eye_region

In [9]:
def extract_colors(left_cheek_region, right_cheek_region, hair_region, eye_region):

    # cheek:
    lower_skin = (0, 40, 60)
    upper_skin = (25, 200, 255)

    # left
    hsv_cheek_left = cv2.cvtColor(left_cheek_region, cv2.COLOR_BGR2HSV)
    mask_cheek_left = cv2.inRange(hsv_cheek_left, lower_skin, upper_skin)

    skin_left = left_cheek_region[mask_cheek_left > 0]

    # right
    hsv_cheek_right = cv2.cvtColor(right_cheek_region, cv2.COLOR_BGR2HSV)
    mask_cheek_right = cv2.inRange(hsv_cheek_right, lower_skin, upper_skin)

    skin_right = right_cheek_region[mask_cheek_right > 0]

    
    # hair:
    hsv_hair = cv2.cvtColor(hair_region, cv2.COLOR_BGR2HSV)

    lower_skin = (0,30,60)
    upper_skin = (20,150,255)

    skin_mask = cv2.inRange(hsv_hair, lower_skin, upper_skin)
    hair_mask = cv2.bitwise_not(skin_mask)

    hair_pixels = hair_region[hair_mask > 0]
    hair_only = hair_pixels[np.any(hair_pixels > 10, axis=1)]

    
    # eye:
    lower_eye = (0, 40, 40)
    upper_eye = (180, 255, 200)
    
    hsv_eye = cv2.cvtColor(eye_region, cv2.COLOR_BGR2HSV)
    mask_eye = cv2.inRange(hsv_eye, lower_eye, upper_eye)

    eye_only = eye_region[mask_eye > 0]

    
    return skin_left, skin_right, hair_only, eye_only

## Threads

In [10]:
# face detection algorithm lol
face_cascade = cv2.CascadeClassifier(
    "/home/xilinx/jupyter_notebooks/base/video/data/haarcascade_frontalface_default.xml"
)

def face_thread():
    print("[FACE] Thread starting")
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("[FACE] Camera failed")
        return

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    time.sleep(1)

    while True:
        ret, frame = cap.read()
        if not ret:
            time.sleep(0.5)
            continue

        gray_small = cv2.cvtColor(cv2.resize(frame, (320, 240)), cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(
            gray_small,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(60, 60)
        )

        with lock:
            shared["face_count"] = len(faces)

            # check if led set the capture bit
            if shared.get("capture_request", False):
                shared["captured_frame"] = frame.copy()
                shared["capture_request"] = False
                print("[FACE] Frame captured on request")
                break

        time.sleep(0.05)

    cap.release()
    print("[FACE] Thread exiting")

In [11]:
def led_thread():
    print("[LED] Thread starting")
    current_state = None
    single_face_start = None

    while True:
        with lock:
            count = shared.get("face_count", 0)

        # no face
        if count == 0:
            if current_state != "red":
                set_led_red()
                current_state = "red"
            single_face_start = None

        # more than 1 face
        elif count > 1:
            if current_state != "blue":
                set_led_blue()
                current_state = "blue"
            single_face_start = None

        # exactly 1 face
        elif count == 1:
            if current_state != "green":
                set_led_green()
                current_state = "green"
            if single_face_start is None:
                single_face_start = time.time()
            elif time.time() - single_face_start >= 3:
                print("[LED] Stable face for 3 seconds")

                # indicate about to take picture
                for _ in range(3):
                    clear_gpios()
                    time.sleep(0.2)
                    set_led_green()
                    time.sleep(0.2)

                # set signal bit to camera
                with lock:
                    shared["capture_request"] = True

                write_rgb_led(1,0,1)
                print("[LED] Capture requested, exiting LED thread")
                break
        time.sleep(0.1)

    print("[LED] Thread exiting")

## MAIN

In [ ]:
if __name__ == "__main__":
    clear_gpios()
    
    # saving images set up
    folder_name = input("Enter your name: ").strip()
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
    else:
        # remove images
        for filename in os.listdir(folder_name):
            if filename.lower().endswith(".jpg"):
                file_path = os.path.join(folder_name, filename)
                os.remove(file_path)

    print(f"[INFO] Images will be saved in folder: {folder_name}")
    
    # thread set up
    shared = {
        "face_count": 0,
        "capture_request": False,
        "capture_done": False,
        "captured_frame": None
    }
    frame = None

    lock = threading.Lock()

    t_face = threading.Thread(target=face_thread)
    t_led = threading.Thread(target=led_thread)

    t_face.start()
    t_led.start()

    t_led.join()
    t_face.join()

    print("[MAIN] Threads joined")

# process captured frame
frame = shared.get("captured_frame", None)
timestamp = int(time.time())

if frame is not None:
    cv2.imwrite(os.path.join(folder_name, f"original_picture_{timestamp}.jpg"), frame)
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(60, 60)
    )
    
    if len(faces) > 0:
        x, y, w, h = faces[0]
    
    frame_corrected = color_correct(frame, (x,y,w,h))
    cv2.imwrite(os.path.join(folder_name, f"captured_corrected_{timestamp}.jpg"), frame_corrected)
    print(f"[MAIN] Captured image color-corrected and saved as '{folder_name}/captured_corrected_{timestamp}.jpg'")

    # ------------------------------------------------------------------
    # processing part

    print("[MAIN] Processing captured frame...")
    write_rgb_led(0,1,1)

    frame = frame_corrected

    if len(faces) > 0:
        x, y, w, h = faces[0]
        
        left_cheek, right_cheek, hair, eye = extract_regions(frame, x,y,w,h)
        
        left_cheek_region = left_cheek["region"]
        right_cheek_region = right_cheek["region"]
        hair_region = hair["region"]
        eye_region = eye["region"]

        skin_left, skin_right, hair_only, eye_only = extract_colors(
            left_cheek_region,
            right_cheek_region,
            hair_region,
            eye_region
        )
        
        skin_only = np.vstack((skin_left, skin_right))
        
        # COLORS WOWIE THIS IS THE ONE
        colors = [get_color(skin_only), get_color(hair_only), get_color(eye_only)]
            
        print("Skin Color:", colors[0])
        print("Hair Color:", colors[1])
        print("Eye Color:", colors[2])
        
        # ------------------------------------------------------------------
        # debug images
        cv2.imwrite(os.path.join(folder_name, f"debug_skin_l_{timestamp}.jpg"), left_cheek_region)
        cv2.imwrite(os.path.join(folder_name, f"debug_skin_r_{timestamp}.jpg"), right_cheek_region)
        cv2.imwrite(os.path.join(folder_name, f"debug_hair_{timestamp}.jpg"), hair_region)
        cv2.imwrite(os.path.join(folder_name, f"debug_eye_{timestamp}.jpg"), eye_region)
        
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)

        for region, color in [
            (left_cheek, (255,0,0)),
            (right_cheek, (255,0,0)),
            (hair, (0,0,255)),
            (eye, (255,255,0)),
        ]:
            x1, y1, x2, y2 = region["box"]
            cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)

        cv2.imwrite(os.path.join(folder_name, f"captured_with_boxes_{timestamp}.jpg"), frame)
        print(f"[MAIN] Image saved as {folder_name}/captured_with_boxes_{timestamp}.jpg")

    else:
        print("[MAIN] No face found in captured frame")

else:
    print("[MAIN] No frame captured")

# ------------------------------------------------------------------
# socket time
print("starting socket connection to server")
write_rgb_led(1,1,1)

ip = "192.168.2.1"
port = 9999

print(f"server ip: {ip}, port: {port}")

sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

sock.connect((ip, port))
print(f"Connected to {ip}:{port}")

time.sleep(1)

for c in colors:
    res = ' '.join(str(val) for val in c)
    res = res + '\n'
    sock.sendall(res.encode('utf-8'))
    time.sleep(0.5)

sock.close()

clear_gpios()
print("Done.")

Enter your name: caroline
[INFO] Images will be saved in folder: caroline
[FACE] Thread starting
[LED] Thread starting


[ WARN:7] global ./modules/videoio/src/cap_gstreamer.cpp (616) isPipelinePlaying OpenCV | GStreamer warning: GStreamer: pipeline have not been created
[ WARN:7] global ./modules/videoio/src/cap_v4l.cpp (890) open VIDEOIO(V4L2:/dev/video0): can't open camera by index


[FACE] Camera failed


In [35]:
clear_gpios()